# 02. ChiPseq сигнал в A/B компартментах

**Цель:** сравнить уровень ChiPseq сигнала в A и B компартментах.

В этом блокноте мы не считаем количество пики. Для разных ChiPseq меток это не очень удобно сравнивать напрямую. Вместо этого берем готовые `bw` треки и считаем средний сигнал в каждом компартментном окне.

**Входные файлы:**

- `bw` треки ChiPseq из третьего дня;
- `bedGraph` с A/B компартментами из второго дня.

**Что получится:** таблица со средним ChiPseq сигналом в окнах A/B и скрипичный график для каждой метки.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pyBigWig
from scipy.stats import mannwhitneyu

sns.set_theme(style="whitegrid")

### Пути к файлам

Берем те файлы, которые уже получили на предыдущих практиках.

In [ ]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

compartments_path = ROOT / "../day2_HiC_practice/results/compartments/MoPh7_enr_v2_E1_res1000000.bedGraph"

bw_files = {
    "H2A119Ub": ROOT / "../day3_ChIPseq_practice/results/macs/MoPh7_H2A119Ub/MoPh7_H2A119Ub_FE.bw",
    "H3K9me3": ROOT / "../day3_ChIPseq_practice/results/macs/MoPh7_H3K9me3/MoPh7_H3K9me3_FE.bw",
    "H3K27Ac": ROOT / "../day3_ChIPseq_practice/results/macs/MoPh7_H3K27Ac/MoPh7_H3K27Ac_FE.bw",
}

print("compartments:", compartments_path.exists(), compartments_path)
for mark, path in bw_files.items():
    print(mark, path.exists(), path)

### Читаем A/B компартменты

В `bedGraph` лежит значение собственный вектор. Для простого анализа используем знак:

- `score >= 0` — A компартмент;
- `score < 0` — B компартмент.

In [ ]:
def add_chr_prefix(chrom):
    chrom = str(chrom)
    return chrom if chrom.startswith("chr") else "chr" + chrom


def chromosome_group(chrom):
    chrom = str(chrom)
    human_chroms = {f"chr{i}" for i in range(1, 23)} | {"chrX", "chrY"}
    mosquito_chroms = {"2L", "2R", "3L", "3R", "Xmos", "XMOS", "plm"}

    if chrom in human_chroms:
        return "Human"
    if chrom in mosquito_chroms:
        return "Mosquito"
    return "Other"

In [ ]:
compartments = pd.read_csv(
    compartments_path,
    sep="\t",
    names=["chrom", "start", "end", "score"],
)

compartments["chrom"] = compartments["chrom"].map(add_chr_prefix)
compartments["compartment"] = np.where(compartments["score"] >= 0, "A", "B")
compartments["species"] = compartments["chrom"].map(chromosome_group)
compartments["group"] = compartments["species"] + " " + compartments["compartment"]

compartments.head()

In [ ]:
compartments["group"].value_counts()

### Достаем средний сигнал из `bw`

Для каждого окна компартмента берем средний ChiPseq сигнал из `bw`.

Если в `bw` нет нужной хромосомы, такое окно пропускаем.

In [ ]:
def mean_bw_signal(bw, chrom, start, end):
    if chrom not in bw.chroms():
        return np.nan

    chrom_size = bw.chroms()[chrom]
    start = max(0, int(start))
    end = min(int(end), chrom_size)

    if end <= start:
        return np.nan

    value = bw.stats(chrom, start, end, type="mean")[0]
    return value

In [ ]:
def collect_signal_for_mark(mark, bw_path, compartments):
    rows = []

    with pyBigWig.open(str(bw_path)) as bw:
        for row in compartments.itertuples(index=False):
            value = mean_bw_signal(bw, row.chrom, row.start, row.end)

            if np.isnan(value):
                continue

            rows.append({
                "mark": mark,
                "chrom": row.chrom,
                "start": row.start,
                "end": row.end,
                "compartment": row.compartment,
                "species": row.species,
                "group": row.group,
                "value": value,
            })

    return pd.DataFrame(rows)


signal_tables = []

for mark, bw_path in bw_files.items():
    table = collect_signal_for_mark(mark, bw_path, compartments)
    signal_tables.append(table)
    print(mark, table.shape)

signal_df = pd.concat(signal_tables, ignore_index=True)
signal_df.head()

### Средние значения по группам

Перед графиком полезно посмотреть таблицу: сколько окон попало в каждую группу и какой средний сигнал.

In [ ]:
summary = (
    signal_df
    .groupby(["mark", "group"], as_index=False)
    .agg(
        n_windows=("value", "size"),
        mean_signal=("value", "mean"),
        median_signal=("value", "median"),
    )
)

summary

### Violin plot

Каждая точка распределения — это одно окно компартмента.

Для каждой ChiPseq метки сравниваем сигнал в A и B компартментах. Если в данных есть human и mosquito хромосомы, они будут показаны отдельно.

In [ ]:
def format_pvalue(pvalue):
    if np.isnan(pvalue):
        return "p=NA"
    if pvalue < 1e-4:
        return "p<1e-4"
    if pvalue < 1e-2:
        return f"p={pvalue:.1e}"
    return f"p={pvalue:.3f}"


def mannwhitney_pvalue(data, group_a, group_b):
    a = data.loc[data["group"] == group_a, "value"].dropna()
    b = data.loc[data["group"] == group_b, "value"].dropna()

    if len(a) == 0 or len(b) == 0:
        return np.nan

    return mannwhitneyu(a, b, alternative="two-sided").pvalue

In [ ]:
group_order = [
    "Human A",
    "Human B",
    "Mosquito A",
    "Mosquito B",
]
group_order = [group for group in group_order if group in signal_df["group"].unique()]

palette = {
    "Human A": "#762a83",
    "Human B": "#c2a5cf",
    "Mosquito A": "#1b7837",
    "Mosquito B": "#a6dba0",
}

fig, axes = plt.subplots(1, len(bw_files), figsize=(5.5 * len(bw_files), 5), sharey=False)

if len(bw_files) == 1:
    axes = [axes]

for ax, mark in zip(axes, bw_files):
    plot_data = signal_df[signal_df["mark"] == mark].copy()

    sns.violinplot(
        data=plot_data,
        x="group",
        y="value",
        hue="group",
        order=group_order,
        palette=palette,
        inner="box",
        cut=0,
        legend=False,
        ax=ax,
    )

    labels = []
    for group in group_order:
        mean_value = plot_data.loc[plot_data["group"] == group, "value"].mean()
        labels.append(f"{group}\n(mean={mean_value:.2f})")

    ax.set_xticks(range(len(group_order)))
    ax.set_xticklabels(labels)
    ax.set_title(mark)
    ax.set_xlabel("группа")
    ax.set_ylabel("сигнал")

    y_min, y_max = ax.get_ylim()
    text_y = y_max - (y_max - y_min) * 0.08

    for species in ["Human", "Mosquito"]:
        group_a = f"{species} A"
        group_b = f"{species} B"
        if group_a in group_order and group_b in group_order:
            pvalue = mannwhitney_pvalue(plot_data, group_a, group_b)
            x_position = (group_order.index(group_a) + group_order.index(group_b)) / 2
            ax.text(x_position, text_y, format_pvalue(pvalue), ha="center", va="top", fontsize=11)

plt.suptitle("MoPh7", fontsize=16)
plt.tight_layout()
plt.show()


### Как интерпретировать

Этот график показывает не количество ChiPseq пики, а уровень сигнала в `bw`.

Если сигнал выше в A компартменте, это может говорить о связи метки с более активной пространственной организацией хроматина. Если сигнал выше в B компартменте, метка может быть связана с более плотными или репрессивными областями.

Важно: это простое сравнение распределений, а не доказательство причинной связи.